# Dataset Integration: Epoch Database + Open LLM Leaderboard

This notebook builds the project dataset by combining two complementary sources of information about language models. The objective is to create a single table that contains both technical/model metadata and benchmark performance results.

## Data sources

Two datasets are used in this notebook:

**1. Epoch Database - Notable Models**  
This source contains technical and contextual information about notable AI models, such as the organization, publication date, parameter count, training compute, training dataset, accessibility, country, and other metadata. It is useful because it provides information about how each model was built and released.

**2. Open LLM Leaderboard formatted dataset**  
This source contains evaluation results for models available on Hugging Face, including average benchmark scores and individual benchmark results such as IFEval, BBH, MATH, GPQA, MuSR, and MMLU-Pro. It is useful because it provides the performance variables that the project later analyses or predicts.

The two sources are heterogeneous: they do not use exactly the same model names, and some Hugging Face entries correspond to variants, fine-tunes, or versions of a base model. For this reason, the merge is performed using normalized model-name keys rather than exact raw names.

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
epoch_df = pd.read_csv('../data/raw/Epoch Database - Notable Models.csv')
leaderboard_df = pd.read_csv('../data/raw/formatted.csv')

print(f"Epoch dataset: {epoch_df.shape[0]} rows × {epoch_df.shape[1]} columns")
print(f"Leaderboard dataset: {leaderboard_df.shape[0]} rows × {leaderboard_df.shape[1]} columns")

Epoch dataset: 981 rows × 47 columns
Leaderboard dataset: 4576 rows × 40 columns


## Matching strategy

The main difficulty is that the same model can appear with different names in each source. For example, one source may include the organization prefix, while the other may include suffixes such as `-Instruct`, `-Chat`, `-HF`, quantization labels, or version tags.

To reduce this problem, model names are transformed into simplified matching keys. The process:

1. Converts names to lowercase.
2. Removes the creator prefix when names follow the Hugging Face format `organization/model-name`.
3. Removes frequent suffixes and labels that often prevent equivalent models from matching.
4. Removes version tags and special characters.

This is an approximate matching strategy. It increases the number of useful matches, but it can also merge some variants that are not perfectly identical. Since the project focuses on broad trends rather than exact one-to-one model comparison, this limitation is acceptable as long as it is acknowledged in the methodology.

In [3]:
def normalize_model_name(model_name):
    model_name = str(model_name).lower()
    model_name = model_name.split('/')[-1]

    noisy_tokens = [
        '-hf', 'hf', '-instruct', 'instruct', '-chat', 'chat', '-base', 'base',
        '-preview', 'preview', 'awq', 'gptq', 'gguf', 'fp16', 'slerp', 'merge',
        'meta', 'google', 'microsoft', 'mistral'
    ]

    for token in noisy_tokens:
        model_name = model_name.replace(token, '')

    model_name = re.sub(r'(-?v\d+(\.\d+)?)', '', model_name)
    model_name = re.sub(r'[^a-z0-9]', '', model_name)

    return model_name

## Dataset union

Two complementary joins are performed:

**Direct model-name match:** compares the normalized Epoch model name with the normalized Hugging Face model name.

**Base-model match:** compares the normalized Epoch model name with the normalized Hugging Face base model. This helps recover benchmark rows where the evaluated model is a variant or fine-tuned version of a known base model.

After both joins, the results are concatenated and duplicated model pairs are removed.

In [4]:
epoch_df['model_key'] = epoch_df['Model'].apply(normalize_model_name)
leaderboard_df['model_key_name'] = leaderboard_df['model.name'].apply(normalize_model_name)
leaderboard_df['model_key_base'] = leaderboard_df['metadata.base_model'].apply(normalize_model_name)

direct_matches_df = pd.merge(
    epoch_df,
    leaderboard_df,
    left_on='model_key',
    right_on='model_key_name',
    how='inner'
)

base_model_matches_df = pd.merge(
    epoch_df,
    leaderboard_df,
    left_on='model_key',
    right_on='model_key_base',
    how='inner'
)

merged_models_df = (
    pd.concat([direct_matches_df, base_model_matches_df])
    .drop_duplicates(subset=['model.name', 'Model'])
    .reset_index(drop=True)
)

print(f"Direct matches: {len(direct_matches_df)}")
print(f"Base-model matches: {len(base_model_matches_df)}")
print(f"Final merged rows after deduplication: {len(merged_models_df)}")

merged_models_df.to_excel('../data/interim/modelos_unidos.xlsx', index=False)
print(f"Merged dataset exported to: modelos_unidos.xlsx")

Direct matches: 34
Base-model matches: 75
Final merged rows after deduplication: 77
Merged dataset exported to: modelos_unidos.xlsx


## Final dataset selection

The intermediate merged file contains many columns from both original sources. For the final project dataset, only the most useful variables are kept:

- model identity and organization information;
- publication and accessibility metadata;
- technical characteristics such as parameters, training compute, dataset size, architecture, and precision;
- benchmark scores from the Open LLM Leaderboard;
- matching keys used to trace how the merge was produced.

Column names are converted to a consistent English `snake_case` format to make the dataset easier to use in the following analysis notebooks.

In [ ]:
merged_models_df = pd.read_excel('../data/interim/modelos_unidos.xlsx')

column_mapping = {
    'Model': 'model',
    'Organization': 'organization',
    'Publication date': 'publication_date',
    'Domain': 'domain',
    'Task': 'task',
    'Parameters': 'parameters',
    'Training compute (FLOP)': 'training_flops',
    'Training dataset': 'training_dataset',
    'Training dataset size (total)': 'dataset_size',
    'Confidence': 'confidence',
    'Link': 'link',
    'Reference': 'reference',
    'Organization categorization': 'organization_type',
    'Country (of organization)': 'country',
    'Notability criteria': 'notability_criteria',
    'Epochs': 'epochs',
    'Base model': 'base_model',
    'Model accessibility': 'model_accessibility',
    'Training code accessibility': 'training_code_accessibility',
    'Numerical format': 'numerical_format',
    'Open model weights?': 'open_weights',
    'model.name': 'model_hf',
    'model.precision': 'precision',
    'model.type': 'model_type',
    'model.weight_type': 'weight_type',
    'model.architecture': 'architecture',
    'model.average_score': 'average_score',
    'model.has_chat_template': 'has_chat_template',
    'evaluations.ifeval.name': 'ifeval_name',
    'evaluations.ifeval.value': 'ifeval',
    'evaluations.ifeval.normalized_score': 'ifeval_normalized',
    'evaluations.bbh.name': 'bbh_name',
    'evaluations.bbh.value': 'bbh',
    'evaluations.bbh.normalized_score': 'bbh_normalized',
    'evaluations.math.name': 'math_name',
    'evaluations.math.value': 'math',
    'evaluations.math.normalized_score': 'math_normalized',
    'evaluations.gpqa.name': 'gpqa_name',
    'evaluations.gpqa.value': 'gpqa',
    'evaluations.gpqa.normalized_score': 'gpqa_normalized',
    'evaluations.musr.name': 'musr_name',
    'evaluations.musr.value': 'musr',
    'evaluations.musr.normalized_score': 'musr_normalized',
    'evaluations.mmlu_pro.name': 'mmlu_pro_name',
    'evaluations.mmlu_pro.value': 'mmlu_pro',
    'evaluations.mmlu_pro.normalized_score': 'mmlu_pro_normalized',
    'features.is_not_available_on_hub': 'not_available_hub',
    'features.is_moe': 'is_moe',
    'features.is_official_provider': 'official_provider',
    'metadata.upload_date': 'upload_date',
    'metadata.submission_date': 'submission_date',
    'metadata.generation': 'generation',
    'metadata.base_model': 'base_model_hf',
    'metadata.hub_license': 'license',
    'metadata.hub_hearts': 'hub_likes',
    'metadata.params_billions': 'parameters_billions',
    'metadata.co2_cost': 'co2_cost',
    'model_key': 'model_key',
    'id': 'id',
    'model_key_name': 'model_key_name',
    'model_key_base': 'model_key_base'
}

final_df = merged_models_df.rename(columns=column_mapping)

selected_columns = [
    'model', 'organization', 'publication_date', 'domain', 'task', 'parameters',
    'training_flops', 'training_dataset', 'dataset_size', 'confidence', 'link',
    'reference', 'organization_type', 'country', 'notability_criteria', 'epochs',
    'base_model', 'model_accessibility', 'training_code_accessibility',
    'numerical_format', 'open_weights', 'model_key', 'id', 'model_hf', 'precision',
    'model_type', 'weight_type', 'architecture', 'average_score', 'has_chat_template',
    'ifeval_name', 'ifeval', 'ifeval_normalized', 'bbh_name', 'bbh',
    'bbh_normalized', 'math_name', 'math', 'math_normalized', 'gpqa_name', 'gpqa',
    'gpqa_normalized', 'musr_name', 'musr', 'musr_normalized', 'mmlu_pro_name',
    'mmlu_pro', 'mmlu_pro_normalized', 'not_available_hub', 'is_moe',
    'official_provider', 'upload_date', 'submission_date', 'generation',
    'base_model_hf', 'license', 'hub_likes', 'parameters_billions', 'co2_cost',
    'model_key_name', 'model_key_base'
]

final_df = final_df[[column for column in selected_columns if column in final_df.columns]]

print(f"Final dataset: {final_df.shape[0]} rows × {final_df.shape[1]} columns")

final_df.to_excel('../data/processed/final_dataset.xlsx', index=False)
print(f"Final dataset exported to: ../data/processed/final_dataset.xlsx")

Final dataset: 77 rows × 61 columns
Final dataset exported to: final_dataset.xlsx


## Methodological limitation

The merge is based on normalized model names rather than a universal model identifier. This is necessary because the two data sources use different naming conventions. However, it means that some matches may correspond to model variants rather than exactly identical models.

For this reason, the resulting dataset should be interpreted as an exploratory integration of technical and benchmark information. It is suitable for identifying general patterns, but not for making very strict claims about individual model rankings or exact model-to-model equivalence.